In [16]:
import torch 
import torch.nn as nn
import torch.nn.functional as F

class FC_Attention(nn.Module):
    def __init__(
        self,
        embed_dim=256,
        hidden_dim=512,
        q_dim=512,
        v_dim=256,
        num_heads=8,
        dropout=0.0,
        block_index=0,
        internal_resolution=(32, 32),
        query_projection_kernel_size=1,
        key_projection_kernel_size=3,
        value_projection_kernel_size=3,
        kv_kernel_size=3,
        head_unification_kernel_size=3,
        query_projection_stride=1,
        key_projection_stride=1,
        value_projection_stride=1,
        kv_stride=1,
        head_unification_stride=1,
        query_projection_padding=0,
        key_projection_padding=1,
        value_projection_padding=1,
        kv_padding=1,
        head_unification_padding=1,
        query_projection_dilation_factor=0,
        key_projection_dilation_factor=0,
        value_projection_dilation_factor=0,
        kv_dilation_factor=0,
        head_unification_dilation_factor=0,
        use_attention_bias=True,
    ):
        super().__init__()
        self.q_net = nn.Conv2d(
            embed_dim,
            q_dim,
            kernel_size=query_projection_kernel_size,
            padding=query_projection_padding,
            stride=query_projection_stride,
            dilation=int(internal_resolution[0] * query_projection_dilation_factor) if query_projection_dilation_factor else 1,
        )
        self.k_net = nn.Conv2d(
            embed_dim,
            q_dim,
            kernel_size=key_projection_kernel_size,
            stride=key_projection_stride,
            padding=key_projection_padding,
            dilation=int(internal_resolution[0] * key_projection_dilation_factor) if key_projection_dilation_factor else 1,
        )
        self.v_net = nn.Conv2d(
            embed_dim,
            v_dim,
            kernel_size=value_projection_kernel_size,
            stride=value_projection_stride,
            padding=value_projection_padding,
            dilation=(
                int(internal_resolution[0] * value_projection_dilation_factor) if value_projection_dilation_factor else 1
            ),
        )
        if use_attention_bias:
            self.bias_net = nn.Linear(embed_dim, v_dim)
        if num_heads > 1:
            self.head_unification = nn.Conv2d(
                v_dim,
                embed_dim,
                kernel_size=head_unification_kernel_size,
                padding=head_unification_padding,
                stride=head_unification_stride,
                dilation=(
                    int(internal_resolution[0] * head_unification_dilation_factor) if head_unification_dilation_factor else 1
                ),
            )
        self.num_heads = num_heads
        self.hidden_dim = hidden_dim
        self.q_dim = q_dim
        self.v_dim = v_dim
        self.embed_dim = embed_dim
        self.block_index = block_index
        self.internal_resolution = internal_resolution
        self.use_attention_bias = use_attention_bias
        self.kv_kernel_size = kv_kernel_size
        self.kv_stride = kv_stride
        self.kv_padding = kv_padding
        self.kv_dilation_factor = kv_dilation_factor

    def break_into_heads(self, M):
        B, D, H, W = M.shape
        h = self.num_heads
        return M.reshape(B, h, D // h, H, W).reshape(B * h, D // h, H, W)

    def sum_pool_to_resolution(self, x, output_resolution=3):
        a = F.interpolate(x, size=(output_resolution, output_resolution), mode="bilinear")
        return a

    def phi(self, x, p=2):
        x = F.relu(x)
        xp = x**p
        numerator = torch.norm(x) * xp
        denominator = torch.norm(xp)
        return numerator / denominator

    def spatial_FLatten_attention(self, x):
        Q = self.q_net(x)
        K = self.k_net(x)
        V = self.v_net(x)
        print("Q shape:", Q.shape)
        print("K shape:", K.shape)
        print("V shape:", V.shape)
        print("=================================")

        B, Dq, H, W = Q.shape
        _, Dv, _, _ = V.shape
        _, Dm, _, _ = x.shape
        h = self.num_heads

        phi_Q = self.phi(Q)
        phi_K = self.phi(K)

        phi_Q = self.break_into_heads(phi_Q)
        phi_K = self.break_into_heads(phi_K)
        V = self.break_into_heads(V)

        print("phi_Q shape after breaking into heads:", phi_Q.shape)
        print("phi_K shape after breaking into heads:", phi_K.shape)
        print("V shape after breaking into heads:", V.shape)

        print("=================================")
        KV = phi_K.unsqueeze(2) * V.unsqueeze(1)
        print("KV shape after outer product:", KV.shape)
        KV = KV.reshape(-1, *KV.shape[2:])
        print("KV shape after reshaping for pooling:", KV.shape)
        KV = self.sum_pool_to_resolution(KV, output_resolution=self.kv_kernel_size)
        print("KV shape after pooling:", KV.shape)

        abs_max = 32 / (self.kv_kernel_size**2)
        KV = KV.clamp(min=-abs_max, max=abs_max)

        print("=================================")

        KV = KV.reshape(B * h, Dq // h, Dv // h, self.kv_kernel_size, self.kv_kernel_size)
        print("KV shape after reshaping for convolution:", KV.shape)
        KV = KV.permute(0, 2, 1, 3, 4)
        print("KV shape after permuting for convolution:", KV.shape)
        KV = KV.reshape(-1, *KV.shape[2:])
        print("KV shape after reshaping for convolution:", KV.shape)
        print("=================================")

        # Reshape Q into a single B * Dq channel image
        phi_Q = phi_Q.reshape(-1, *phi_Q.shape[2:]).unsqueeze(0)
        print("phi_Q shape after reshaping for convolution:", phi_Q.shape)
        bias = None
        if self.use_attention_bias:
            # Squeeze and excite bias
            squeezed = x.mean(dim=(2, 3))
            excited = self.bias_net(squeezed)
            bias = excited.reshape(-1)

        print("=================================")
        print("BEFORE CONVOLUTION:")
        print("phi_Q shape:", phi_Q.shape)
        print("KV shape:", KV.shape)
        # QKV is grouped (B*h groups) convolution of Q with KV
        QKV = F.conv2d(
            phi_Q,
            KV,
            bias=bias,
            groups=B * h,
            padding=self.kv_padding,
            stride=self.kv_stride,
            dilation=int(self.internal_resolution[0] * self.kv_dilation_factor) if self.kv_dilation_factor else 1,
        )
        print("\nAFTER CONVOLUTION:")
        print("QKV shape after convolution:", QKV.shape)
        
        QKV = QKV.view(B, Dv, H, W)
        print("QKV shape after reshaping back to (B, Dv, H, W):", QKV.shape)

        # Unify heads
        if h > 1:
            QKV = self.head_unification(QKV)
        
        print("QKV shape after head unification:", QKV.shape)
        QKV = F.layer_norm(QKV, QKV.shape[1:])
        print("QKV shape after layer normalization:", QKV.shape)
        return QKV

    def forward(self, x):
        B, D, H, W = x.shape
        attn = self.spatial_FLatten_attention(x)
        return attn


In [17]:
ex = torch.randn(1, 256, 32, 32)
model = FC_Attention()
out = model(ex)
print(out.shape)

Q shape: torch.Size([1, 512, 32, 32])
K shape: torch.Size([1, 512, 32, 32])
V shape: torch.Size([1, 256, 32, 32])
phi_Q shape after breaking into heads: torch.Size([8, 64, 32, 32])
phi_K shape after breaking into heads: torch.Size([8, 64, 32, 32])
V shape after breaking into heads: torch.Size([8, 32, 32, 32])
KV shape after outer product: torch.Size([8, 64, 32, 32, 32])
KV shape after reshaping for pooling: torch.Size([512, 32, 32, 32])
KV shape after pooling: torch.Size([512, 32, 3, 3])
KV shape after reshaping for convolution: torch.Size([8, 64, 32, 3, 3])
KV shape after permuting for convolution: torch.Size([8, 32, 64, 3, 3])
KV shape after reshaping for convolution: torch.Size([256, 64, 3, 3])
phi_Q shape after reshaping for convolution: torch.Size([1, 512, 32, 32])
BEFORE CONVOLUTION:
phi_Q shape: torch.Size([1, 512, 32, 32])
KV shape: torch.Size([256, 64, 3, 3])

AFTER CONVOLUTION:
QKV shape after convolution: torch.Size([1, 256, 32, 32])
QKV shape after reshaping back to (B, Dv,